In [22]:
!pip install anthropic python-dotenv

In [23]:
from dotenv import load_dotenv
import json
load_dotenv()

True

In [24]:
from anthropic import Anthropic
from statistics import mean
import ast
import re

client = Anthropic()
model = "claude-haiku-4-5"

messages = []

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 500,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    <task>
    Task: {test_case['task']}
    </task>
    <solution>
    Solution: {output}
    </solution>
    <criteria>
    Criteria to evaluate: {test_case['solution_criteria']}
    </criteria>
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

def grade_syntax(test_case, output):

    format = test_case.get("format")
    if format == "json":
        return validate_json(output)
    elif format == "python":
        return validate_python(output)
    elif format == "regex":
        return validate_regex(output)
    else:
        return 0
def run_prompt(test_case):
    prompt = f"""
    Solve the following task:

    {test_case['task']}

    * Respond only with Python, JSON, or regex based on the specified format.
    * Do not include any explanations or comments, only the solution.
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    return chat(messages, stop_sequences=["```"])

def run_test_case(test_case):
    
    response = run_prompt(test_case)
    
    # grading
    syntax_score = grade_syntax(test_case, response)

    evaluation = grade_by_model(test_case, response)
    model_score = evaluation.get("score", 0)
    reasoning = evaluation.get("reasoning", "")

    return {
        "output": response,
        "test_case": test_case,
        "score": (syntax_score + model_score) / 2,
        "reasoning": reasoning
    }

def run_eval(dataset):
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    print(f"Average score: {mean([r['score'] for r in results])}")
    return results

In [25]:
with open('008-dataset.json', 'r') as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results, indent=2))


Average score: 5.0
[
  {
    "output": "\nimport json\nimport re\n\ndef extract_s3_bucket_ids(template_content):\n    \"\"\"\n    Extract all S3 bucket logical IDs from a CloudFormation template.\n    \n    Args:\n        template_content: CloudFormation template as string (JSON or YAML)\n    \n    Returns:\n        List of logical IDs for S3 buckets\n    \"\"\"\n    # Try parsing as JSON first\n    try:\n        template = json.loads(template_content)\n    except json.JSONDecodeError:\n        # If JSON fails, try YAML parsing\n        try:\n            import yaml\n            template = yaml.safe_load(template_content)\n        except:\n            return []\n    \n    # Extract S3 bucket logical IDs\n    s3_buckets = []\n    \n    if isinstance(template, dict) and 'Resources' in template:\n        resources = template['Resources']\n        for logical_id, resource_config in resources.items():\n            if isinstance(resource_config, dict) and resource_config.get('Type') == 'AWS: